# Modelos de Regressão II — Polynomial Features e Regularização

Neste notebook, exploramos **Modelos de Regressão II** aplicados ao dataset *California Housing*, 
focando em **Polynomial Features** e **Regularização** (Ridge, Lasso e Elastic Net), 
sempre com separação em treino, validação e teste, além de análises em Markdown.

**Etapas**
1. Importar bibliotecas e carregar o dataset  
2. Comparar baseline: regressão linear simples e múltipla  
3. Gerar variáveis polinomiais (PolynomialFeatures) e ajustar modelos lineares  
4. Aplicar **Ridge, Lasso e Elastic Net** para controlar coeficientes  
5. Avaliar todos os modelos em *validation* (MAE, RMSE, R²) e depois em *test*  
6. Visualizar e interpretar os resultados (dispersão y_true × y_pred, curvas polinomiais, coeficientes)  
7. Comparar os desempenhos e discutir overfitting vs. generalização


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Reprodutibilidade
RANDOM_STATE = 42

## 1. Carregar o dataset
Vamos carregar o *California Housing* via `sklearn`. O alvo é `MedHouseVal` (valor mediano das casas em centenas de milhares de dólares).

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
display(df.head())
print(f"Formato: {df.shape}")

## 2. Divisão em treino, validação e teste
Vamos dividir os dados de forma estratificada **apenas no sentido reprodutível por `random_state`** (o alvo é contínuo),
em:
- **Treino:** 60%
- **Validação:** 20%
- **Teste:** 20%

Estratégia: primeiro separamos `train` × `temp` (60/40), depois dividimos `temp` em `val` × `test` (50/50).

In [ ]:
# Vamos focar apenas em MedInc para este exemplo visual
X = df[["MedInc"]].copy()
y = df["MedHouseVal"].copy()

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)
print("Shapes:")
print("  Treino:", X_train.shape)
print("  Validação:", X_val.shape)
print("  Teste:", X_test.shape)

## 3. Modelos A, B e C (treino + validação) 

In [ ]:
# ===== Utilitários =====
def eval_metrics(y_true, y_pred, label):
    return {
        "Modelo": label,
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred),
    }

def plot_comparacao_ytrue_ypred(y_true, preds_dict, title="y_true × y_pred — Validação"):
    plt.figure(figsize=(7, 7))
    # linha ideal
    lims = [min(y_true.min(), *(p.min() for p in preds_dict.values())),
            max(y_true.max(), *(p.max() for p in preds_dict.values()))]
    plt.plot(lims, lims, "k--", linewidth=2, label="Ideal (y = x)")
    # pontos
    for label, y_pred in preds_dict.items():
        plt.scatter(y_true, y_pred, alpha=0.35, label=label)
    plt.xlabel("Valor real (y_val)")
    plt.ylabel("Valor predito")
    plt.title(title)
    plt.legend()
    plt.show()



In [ ]:
# ===== Modelo A — Linear Simples (1 variável: MedInc) =====
X_train_A = X_train  # já são apenas ["MedInc"]
X_val_A   = X_val

lr_A = LinearRegression()
lr_A.fit(X_train_A, y_train)
y_val_pred_A = lr_A.predict(X_val_A)

In [ ]:
# ===== Modelo B — Linear Múltipla (todas as variáveis originais) =====
# Construímos X_full e reaproveitamos os MESMOS índices de treino/val/test
X_full = df.drop(columns=["MedHouseVal"])
X_train_B = X_full.loc[X_train.index]
X_val_B   = X_full.loc[X_val.index]

lr_B = LinearRegression()
lr_B.fit(X_train_B, y_train)
y_val_pred_B = lr_B.predict(X_val_B)

In [ ]:
# ===== Modelo C — Polinomial grau 2 APENAS em MedInc =====
poly = PolynomialFeatures(degree=2, include_bias=False)
# usar DataFrame para preservar nome de coluna e evitar warnings
X_train_C_base = pd.DataFrame(X_train["MedInc"])
X_val_C_base   = pd.DataFrame(X_val["MedInc"])

X_train_C_poly = poly.fit_transform(X_train_C_base)
X_val_C_poly   = poly.transform(X_val_C_base)

# nomes das colunas (para debug/inspeção se quiser)
feature_names_C = poly.get_feature_names_out(input_features=["MedInc"])

lr_C = LinearRegression()
lr_C.fit(X_train_C_poly, y_train)
y_val_pred_C = lr_C.predict(X_val_C_poly)

In [ ]:
# ===== Tabela de comparação (validação) =====
resultados_val = pd.DataFrame([
    eval_metrics(y_val, y_val_pred_A, "A) Linear Simples (MedInc)"),
    eval_metrics(y_val, y_val_pred_B, "B) Linear Múltipla (todas)"),
    eval_metrics(y_val, y_val_pred_C, "C) Polinomial (MedInc, grau 2)"),
])

In [ ]:
resultados_val

### 📊 Análise dos resultados

Na comparação dos modelos no conjunto de **validação**:

- **A) Linear Simples (MedInc)**  
  - R² = **0.45** → explica cerca de 45% da variabilidade do target.  
  - MAE = **0.63** e RMSE = **0.71** → erros relativamente altos.  
  - Serve como **baseline inicial**, mas com limitações claras. <br><br>

- **B) Linear Múltipla (todas as variáveis)**  
  - R² = **0.59** → melhora significativa, explicando quase 60% da variabilidade.  
  - MAE = **0.53** e RMSE = **0.53** → menores que os demais modelos.  
  - É o **melhor desempenho entre os três modelos**, confirmando que incluir várias variáveis relevantes aumenta o poder preditivo. <br><br>

- **C) Polinomial (MedInc, grau 2)**  
  - R² = **0.45**, praticamente igual ao modelo simples.  
  - MAE e RMSE pouco melhores ou até piores que o simples.  
  - Mostra que **incluir apenas MedInc² não trouxe ganho expressivo**, reforçando que usar **mais variáveis explicativas** é mais útil do que apenas curvar uma variável isolada.

---

✅ **Conclusão:**  
O modelo **Linear Múltipla (todas as variáveis)** apresenta desempenho superior, indicando que aproveitar diferentes atributos do dataset é mais eficiente do que apenas adicionar um termo polinomial simples em `MedInc`.  

⚠️ **Observação:**  
O polinômio de grau 2 isolado não trouxe vantagem aqui. Isso abre espaço para discutir duas questões importantes:  
- **Polinômios de maior grau** podem trazer flexibilidade, mas precisam ser usados com cautela (risco de overfitting).  
- **Regularização** (Ridge, Lasso, Elastic Net) será fundamental quando adicionarmos muitos termos polinomiais, para controlar o tamanho dos coeficientes e melhorar a generalização.


## 4. Comparação de Modelos com Parâmetros Padrão

Nesta seção, comparamos quatro modelos de regressão linear aplicados ao dataset *California Housing*:

1. **Linear Regression** (sem penalização)  
2. **Ridge Regression (L2)**  
3. **Lasso Regression (L1)**  
4. **Elastic Net (L1 + L2)**  

👉 Todos foram ajustados usando **os parâmetros padrão** do `scikit-learn`:
- Ridge: `alpha=1.0`  
- Lasso: `alpha=0.1`  
- Elastic Net: `alpha=0.1`, `l1_ratio=0.5`  

**Objetivos:**
- Observar diferenças no desempenho (R², MAE, RMSE).  
- Visualizar o efeito da regularização sobre os **coeficientes**.  
- Discutir como Ridge, Lasso e Elastic Net controlam o tamanho dos coeficientes em comparação com a regressão linear comum.


In [ ]:
# Conjunto completo de features (baseline múltipla)
X_full = df.drop(columns=["MedHouseVal"])
X_train_B = X_full.loc[X_train.index]
X_val_B   = X_full.loc[X_val.index]
X_test_B  = X_full.loc[X_test.index]

In [ ]:
# Pipelines com scaler para comparação justa
models = {
    "Linear":     make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge":      make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    "Lasso":      make_pipeline(StandardScaler(), Lasso(alpha=0.1, random_state=RANDOM_STATE, max_iter=10000)),
    "ElasticNet": make_pipeline(StandardScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=10000)),
}

In [ ]:
results_val = []
y_val_preds = {}
fitted = {}

for name, pipe in models.items():
    pipe.fit(X_train_B, y_train)
    y_pred = pipe.predict(X_val_B)
    y_val_preds[name] = y_pred
    fitted[name] = pipe
    results_val.append(eval_metrics(y_val, y_pred, name))

df_results_val = pd.DataFrame(results_val).sort_values("RMSE")
df_results_val

### 📊 Análise dos resultados — Comparação de Modelos com Parâmetros Padrão

No conjunto de **validação**, os modelos apresentaram o seguinte comportamento:

- **Linear Regression (sem penalização)**  
  - R² = **0.592**  
  - MAE = **0.534**  
  - RMSE = **0.533**  
  - Serve como baseline.  
  - Mostra bom ajuste, mas sem controle de coeficientes. <br><br>

- **Ridge Regression (L2)**  
  - R² = **0.592** (praticamente idêntico ao Linear).  
  - MAE e RMSE também iguais ao Linear.  
  - Indica que, com `alpha=1.0`, o efeito da penalização foi pequeno, mas já garante mais **estabilidade dos coeficientes** sem perda de desempenho.  
  - Em cenários com polinômios ou muitas variáveis correlacionadas, o Ridge tende a ser superior. <br><br>

- **Lasso Regression (L1)**  
  - R² = **0.477** → piora significativa.  
  - MAE e RMSE maiores.  
  - O Lasso forçou alguns coeficientes a zero, mas o parâmetro `alpha=0.1` pode ter sido muito restritivo para este caso, reduzindo a capacidade explicativa.  
  - Mostra bem a característica de **seleção de variáveis**, mas neste dataset não trouxe ganho. <br><br>

- **Elastic Net (L1 + L2)**  
  - R² = **0.512** → intermediário, mas ainda abaixo do Linear/Ridge.  
  - Erros maiores do que Linear/Ridge.  
  - Mostra a combinação de encolhimento + seleção, mas aqui também sofreu com o parâmetro padrão (`alpha=0.1`, `l1_ratio=0.5`).

---

✅ **Conclusão:**  
- **Linear e Ridge** tiveram os melhores desempenhos, com resultados equivalentes no conjunto de validação.  
- **Ridge** oferece vantagem conceitual: controla coeficientes sem perda de acurácia.  
- **Lasso e Elastic Net**, com os parâmetros padrão, não performaram bem neste dataset.  
- Isso reforça a importância de **ajustar hiperparâmetros** (`alpha`, `l1_ratio`) via validação cruzada para extrair o melhor dessas técnicas.

👉 Próximo passo natural: aplicar **PolynomialFeatures (grau=2 em todas as variáveis)** e repetir a comparação, onde a regularização deve mostrar ganhos mais claros.


In [ ]:
# Extrair coeficientes das pipelines (último passo é o estimador)
coef_dict = {}
for name, pipe in fitted.items():
    est = pipe.named_steps[list(pipe.named_steps.keys())[-1]]  # último passo
    coef_dict[name] = est.coef_

coefs_df = pd.DataFrame(coef_dict, index=X_train_B.columns)

# (Opcional) limitar às 15 features com maior média de |coef| para legibilidade
top_n = 15
order = coefs_df.abs().mean(axis=1).sort_values(ascending=False).head(top_n).index
coefs_df.loc[order].plot(kind="bar", figsize=(12,6))
plt.axhline(0, color="k", linewidth=0.8)
plt.title("Coeficientes (após padronização) — Linear vs Ridge/Lasso/Elastic Net")
plt.ylabel("Coeficiente")
plt.tight_layout()
plt.show()

coefs_df.loc[order].round(4)

### 📉 Análise da comparação dos coeficientes

O gráfico mostra os coeficientes (após padronização) para cada variável nos quatro modelos:

- **Linear (azul)**  
  - Coeficientes livres, alguns bem altos (ex.: `MedInc` positivo, `Latitude` e `Longitude` negativos).  
  - Nenhum controle, o modelo usa tudo que pode para ajustar. <br><br>

- **Ridge (laranja)**  
  - Coeficientes quase idênticos ao Linear, mas levemente **encolhidos**.  
  - Nenhum coeficiente é zerado, mas todos ficam “mais contidos”.  
  - Indica maior **estabilidade** sem perder informação. <br><br>

- **Lasso (verde)**  
  - Vários coeficientes foram reduzidos, alguns próximos de zero.  
  - Isso mostra o efeito de **seleção de variáveis**: o Lasso “desliga” algumas features consideradas pouco relevantes.  
  - Justifica a queda de desempenho vista na tabela de métricas. <br><br>

- **Elastic Net (vermelho)**  
  - Combina os efeitos: encolhe como o Ridge, mas também zera/reduz alguns como o Lasso.  
  - Resultado intermediário tanto em coeficientes quanto em desempenho.

---

✅ **Conclusão didática:**  
- **Linear:** livre, mas arriscado a overfitting.  
- **Ridge:** suaviza coeficientes, bom para cenários com multicolinearidade.  
- **Lasso:** faz seleção automática de variáveis, mas pode cortar informação útil se o parâmetro não estiver bem ajustado.  
- **Elastic Net:** equilíbrio entre os dois.  

👉 Esse gráfico é essencial para os alunos perceberem que a **regularização não muda só as métricas, mas a própria forma do modelo**: ele passa a usar as variáveis de modo diferente.


## 5. Polynomial Features (grau 2 em todas as variáveis) + Regularização

Até aqui trabalhamos com regressão linear simples, múltipla e com polinômios em apenas uma variável.  
Agora vamos dar um passo além: aplicar **PolynomialFeatures de grau 2 em todas as variáveis** do dataset *California Housing*.  

Isso significa que, além das variáveis originais, o modelo terá também:
- **Termos quadráticos** (ex.: `MedInc²`, `AveRooms²`);  
- **Interações entre variáveis** (ex.: `MedInc × AveRooms`, `Latitude × Longitude`).  

⚠️ O número de variáveis cresce rapidamente: de 8 variáveis originais passamos para **44 variáveis** no grau 2.  
Essa explosão aumenta muito a flexibilidade do modelo, mas também eleva o risco de **overfitting**.

---

### 📌 Objetivos da seção
1. Comparar o desempenho da regressão linear com e sem regularização neste cenário mais complexo.  
2. Avaliar os quatro modelos:  
   - Linear Regression (baseline, sem penalização);  
   - Ridge Regression (L2);  
   - Lasso Regression (L1);  
   - Elastic Net (L1 + L2).  
3. Observar os resultados em termos de:  
   - Métricas de validação (R², MAE, RMSE);  
   - Dispersão `y_true × y_pred`;  
   - Coeficientes estimados (quem permanece grande, quem é encolhido ou zerado).

---

### ✅ O que esperamos observar
- **Linear (sem regularização):** pode apresentar instabilidade e coeficientes muito grandes.  
- **Ridge:** deve suavizar coeficientes sem zerar variáveis.  
- **Lasso:** deve zerar alguns coeficientes, funcionando como seleção automática de variáveis.  
- **Elastic Net:** equilíbrio entre redução e seleção, geralmente mais robusto em cenários complexos.

---


In [ ]:
# ===== Gerar Polynomial Features (grau 2) =====
poly = PolynomialFeatures(degree=2, include_bias=False)

# Dataset expandido (todas as features + polinômios de grau 2)
X_train_poly = poly.fit_transform(X_train_B)
X_val_poly   = poly.transform(X_val_B)
X_test_poly  = poly.transform(X_test_B)

feature_names_poly = poly.get_feature_names_out(X_train_B.columns)
print("Número de variáveis originais:", X_train_B.shape[1])
print("Número de variáveis após grau=2:", X_train_poly.shape[1])

In [ ]:
# Modelos com parâmetros padrão
models_poly = {
    "Linear":     make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge":      make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    "Lasso":      make_pipeline(StandardScaler(), Lasso(alpha=0.1, random_state=RANDOM_STATE, max_iter=10000)),
    "ElasticNet": make_pipeline(StandardScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=10000)),
}

In [ ]:
results_val_poly = []
y_val_preds_poly = {}
fitted_poly = {}

for name, model in models_poly.items():
    model.fit(X_train_poly, y_train)
    y_pred = model.predict(X_val_poly)
    y_val_preds_poly[name] = y_pred
    fitted_poly[name] = model
    results_val_poly.append(eval_metrics(y_val, y_pred, name))

df_results_val_poly = pd.DataFrame(results_val_poly).sort_values("RMSE")
df_results_val_poly

### 📊 Análise dos resultados — Polynomial Features (grau 2 em todas as variáveis)

No conjunto de **validação**, os modelos tiveram o seguinte desempenho:

- **Linear Regression (sem regularização)**  
  - R² = **0.625**, explicando ~62% da variabilidade.  
  - MAE = **0.474** e RMSE = **0.490**.  
  - Apesar do bom ajuste, sofre com o excesso de variáveis, o que aumenta o risco de overfitting. <br><br>

- **Ridge Regression (L2)**  
  - R² = **0.651**, melhor resultado entre os quatro modelos.  
  - MAE = **0.491**, RMSE = **0.456** (menor erro).  
  - Mostra como o **controle dos coeficientes** ajuda o modelo a generalizar melhor.  
  - ✅ **É o vencedor nesta configuração**. <br><br>

- **Lasso Regression (L1)**  
  - R² = **0.488**, bem inferior ao Linear e Ridge.  
  - MAE = **0.617** e RMSE = **0.668**, erros altos.  
  - Como esperado, o Lasso zerou vários coeficientes, mas aqui isso reduziu demais a capacidade explicativa. <br><br>

- **Elastic Net (L1 + L2)**  
  - R² = **0.524**, desempenho intermediário.  
  - MAE = **0.588**, RMSE = **0.622**.  
  - Equilibra shrinkage e seleção de variáveis, mas com os parâmetros padrão (`alpha=0.1`, `l1_ratio=0.5`) não foi tão eficaz quanto o Ridge.

---

✅ **Conclusão:**  
- O **Ridge** se destacou como a melhor escolha neste cenário de polinômios de grau 2, pois conseguiu reduzir o erro e aumentar o R² em relação ao Linear.  
- O **Linear** ainda obteve um bom desempenho, mas sem regularização tende a ser instável quando as variáveis aumentam.  
- **Lasso** e **Elastic Net** não performaram bem com os parâmetros padrão — reforçando a importância de **ajustar hiperparâmetros** (via validação cruzada) para extrair o melhor dessas técnicas.

👉 Esse resultado ilustra muito bem que **regularização não é apenas teoria**: ela realmente faz diferença quando o número de variáveis cresce.

In [ ]:
# helper: pega o último estimador do pipeline (Linear/Ridge/Lasso/ElasticNet)
def last_estimator(pipe):
    # pega o último passo, qualquer que seja o nome
    return pipe.named_steps[list(pipe.named_steps.keys())[-1]]

# DataFrame de coeficientes alinhado aos nomes das features polinomiais
coefs_poly = pd.DataFrame(
    {name: last_estimator(pipe).coef_ for name, pipe in fitted_poly.items()},
    index=feature_names_poly
)

In [ ]:
top_n = 10
order_poly = coefs_poly.abs().mean(axis=1).sort_values(ascending=False).head(top_n).index

ax = coefs_poly.loc[order_poly].plot(kind="bar", figsize=(14, 6))
plt.axhline(0, color="k", linewidth=0.8)
plt.title("Coeficientes (grau 2) — Linear vs Ridge/Lasso/Elastic Net")
plt.ylabel("Coeficiente")
plt.tight_layout()
plt.show()

#coefs_poly.loc[order_poly].round(4)

# Tarefas — Atividade 4 (Moodle)

📌 Este notebook deve ser entregue no Moodle em:  
**Atividade 4 – Modelos de Regressão II (Polynomial Features + Regularização)**

⏰ **Prazo de entrega:** até **hoje, às 23h55**.  
Após esse horário, o sistema permanecerá aberto para envio até **24/09**, porém será aplicada **penalização de nota por atraso (1.4 por dia)**.

---

Realize as análises abaixo. Cada tarefa deve conter:
- **Código em Python** para gerar o resultado.  
- **Markdown explicativo** abaixo do código, comentando os gráficos ou valores obtidos.
---

## Avaliação final no **conjunto de teste**

**Objetivo.** Escolher um modelo com base na **validação** (Seção 5), re-treinar no conjunto **treino+validação** e avaliar no **teste**.  

### Passos
1) **Seleção do melhor modelo pela validação**  
   - Critério principal: **menor RMSE** em *validação*; use MAE e R² como desempate.  
2) **Re-treino** do modelo escolhido em **treino+validação** (80%) com os **mesmos hiperparâmetros**.  
3) **Avaliação no teste (20%)**  
   - Reporte **R², MAE, RMSE**.
4) **Interpretação** (Markdown)  
   - O desempenho no **teste** ficou **coerente** com a validação?  
   - Há sinais de **over/underfitting**?
   - Comente, em 5–8 linhas, por que a **regularização** ajudou (ou não) neste cenário.

> **Reprodutibilidade:** defina `RANDOM_STATE`.

---